In [1]:
import pandas as pd
from collections import defaultdict
from io import StringIO
import json
import sys
import os
# print(os.getcwd())
sys.path.append("g:/programming/Work/proteinNetworkSight/for_work/code/proteinnetworksight")
from DB.scripts.updateDB_tools import open_conn
from DB.scripts.updateDB_tools import get_reverse_drugs_map
from yaml import safe_load
import requests

In [2]:
with open("../../DB/Schemas_new/items/drugs/sources.yaml", "r") as f:
    cfg = safe_load(f.read())
    print(cfg)

{'sources': {'name': 'George Pantziarka TP53 Trust (Created by Anticancer Fund)', 'url': 'https://data.tp53.org.uk/cancerdrugsdb.txt', 'wanted_columns': ['Targets', 'Product', 'DrugBank ID'], 'reversed': True, 'seperator': ';'}}


In [3]:
url = cfg["sources"]["url"]
response = requests.get(url)
df = pd.read_csv(StringIO(response.text), sep='\t')

In [13]:
# skim_df = df[cfg["sources"]["wanted_columns"]]

print(df.columns)
print(df.head(2))

target_product_map = get_reverse_drugs_map(df, seperator=cfg["sources"]["seperator"])
print(target_product_map)

Index(['Product', 'EMA', 'FDA', 'EN', 'Other', 'WHO', 'Year', 'Generic',
       'DrugBank ID', 'ATC', 'ChEMBL', 'Indications', 'Targets', 'Last Update',
       'Unnamed: 14'],
      dtype='object')
       Product EMA FDA EN Other WHO    Year Generic  \
0  Abemaciclib   Y   Y  N   NaN   N  2017.0       N   
1  Abiraterone   Y   Y  N   NaN   Y  2011.0       N   

                                         DrugBank ID      ATC  \
0  <a href="https://go.drugbank.com/drugs/DB12001...  L01EF03   
1  <a href="https://go.drugbank.com/drugs/DB05812...  L02BX03   

                                              ChEMBL  \
0  <a href="https://www.ebi.ac.uk/chembl/compound...   
1  <a href="https://www.ebi.ac.uk/chembl/compound...   

                                        Indications  \
0  Advanced Breast Cancer; Metastatic Breast Cancer   
1   Metastatic Castration Resistant Prostate Cancer   

                                             Targets Last Update  Unnamed: 14  
0  KRAS; ERBB2; EIF4EBP1;

In [6]:
# unique_protein_names = [(name.strip(),) for name in target_product_map.keys()]
unique_protein_names = set([key for key in target_product_map.keys()])
print(unique_protein_names)

{'KDM4E', 'GLS', 'BTRC', 'ABCC4', 'NR2C2', 'TRIM5', 'CD3D', 'CDC42BPA', 'KLRD1', 'TNK2', 'F2', 'MAP2K2', 'SRPK1', 'ABI1', 'SIRT3', 'TBP', 'CASP3', 'F2R', 'COIL', 'ITGB2', 'EPHA4', 'ACTR2', 'PREP', 'CCDC148', 'CYP2C8', 'UHRF1', 'CYP11B1', 'OR10AE3P', 'LGR5', 'SGK1', 'FER', 'NTRK2', 'GPX3', 'UMPS', 'SULT2B1', 'EEF2', 'MAPK10', 'INSR', 'NCOA7', 'PSMC3IP', 'SRI', 'FAP', 'MAN1A1', 'JUN', 'DNMT1', 'RARB', 'PTGDR', 'FOLH1', 'EPHA10', 'COLEC10', 'GSK3B', 'PBRM1', 'GPER1', 'PSMB2', 'LYN', 'CRKL', 'CTAG1B', 'IGFBP3', 'TRPV1', 'MYBBP1A', 'FCGR2B', 'PIK3C2B', 'MPHOSPH8', 'CAPN1', 'TGFBR2', 'TNC', 'RPS6KB1', 'PGR', 'SH2B3', 'IRAK4', 'PPP2R5D', 'PLIN1', 'PMAIP1', 'S100A12', 'FAT1', 'CD22', 'IL7', 'GPR21', 'GLI1', 'TNF', 'AKT3', 'PRKD1', 'NEK2', 'CDKN1B', 'DCP1B', 'CSF2RB', 'PSMA5', 'UGT1A1', 'PMEL', 'VWF', 'NRG3', 'PSMD12', 'FAU', 'S100B', 'SLC28A3', 'GNA11', 'PDGFRB', 'MIR21', 'ISG15', 'ALPP', 'CYP1A1', 'NECTIN4', 'ALOX5AP', 'FASLG', 'MSH5', 'GNL3', 'SYNE3', 'PSMB1', 'CDKN2A', 'EIF4EP1', 'TLR4', 'F

In [7]:
conn = open_conn("../../DB/database.example.ini")
cur = conn.cursor()

g:\programming\Work\proteinNetworkSight\for_work\code\proteinnetworksight\DB\database.example.ini


In [8]:
cur.execute("CREATE TEMP TABLE temp_names(name varchar(50));")
cur.executemany("INSERT INTO temp_names (name) VALUES (%s);", [(name,) for name in unique_protein_names])

In [9]:
sql = """ 
    SELECT protein_id, preferred_name
    FROM items.proteins
    JOIN temp_names n ON n.name = preferred_name;
""" 

cur.execute(sql)

rows = cur.fetchall()

In [10]:
print(rows)

[(6211804, 'SERPINE1'), (6213694, 'BRD4'), (6214165, 'ERBB3'), (6214308, 'ERBB2'), (6214749, 'PDCD4'), (6216226, 'HSPA4'), (6216362, 'FASN'), (6217905, 'STK11'), (6219727, 'FN1'), (6222162, 'SRC'), (6224452, 'GAPDH'), (6224607, 'BCL2'), (6226751, 'BRAF'), (6228101, 'AKT1'), (6230697, 'DIABLO')]


In [11]:
# drug_map = defaultdict(list)
drug_map = {}
print(drug_map)
for row in rows:
    drug_map[row[0]] = target_product_map[row[1]]


print(drug_map)

{}
{6211804: [{'name': 'Alitretinoin', 'drugBankID': 'DB00523'}, {'name': 'Arsenic Trioxide', 'drugBankID': 'DB01169'}, {'name': 'Dexamethasone', 'drugBankID': 'DB01234'}, {'name': 'Epirubicin', 'drugBankID': 'DB00445'}], 6213694: [{'name': 'Docetaxel', 'drugBankID': 'DB01248'}], 6214165: [{'name': 'Afatinib', 'drugBankID': 'DB08916'}, {'name': 'Carboplatin', 'drugBankID': 'DB00958'}, {'name': 'Cetuximab', 'drugBankID': 'DB00002'}, {'name': 'Dacomitinib', 'drugBankID': 'DB11963'}, {'name': 'Docetaxel', 'drugBankID': 'DB01248'}, {'name': 'Erlotinib', 'drugBankID': 'DB00530'}, {'name': 'Gefitinib', 'drugBankID': 'DB00317'}, {'name': 'Lapatinib', 'drugBankID': 'DB01259'}, {'name': 'Momelotinib', 'drugBankID': 'DB11763'}, {'name': 'Pertuzumab', 'drugBankID': 'DB06366'}, {'name': 'Trametinib', 'drugBankID': 'DB08911'}, {'name': 'Trastuzumab', 'drugBankID': 'DB00072'}, {'name': 'Trastuzumab, Hyaluronidase', 'drugBankID': 'DB00072'}, {'name': 'Vandetanib', 'drugBankID': 'DB05294'}, {'name': '

In [12]:
target_product_map['SERPINE1']

[{'name': 'Alitretinoin', 'drugBankID': 'DB00523'},
 {'name': 'Arsenic Trioxide', 'drugBankID': 'DB01169'},
 {'name': 'Dexamethasone', 'drugBankID': 'DB01234'},
 {'name': 'Epirubicin', 'drugBankID': 'DB00445'}]

In [2]:
with open("../../DB/Schemas_new/items/drugs/sources.yaml", "r") as f:
    cfg = safe_load(f.read())
    print(cfg)

for source in cfg["extra"]:
    print(f"{source}: {source}")

    url = source['url']
    print(url)
    # response = requests.get(url)
    # df = pd.read_csv(StringIO(response.text), sep='\t')

    # skim_df = df[cfg["sources"]["wanted_columns"]]

    # target_product_map = get_reverse_drugs_map(skim_df, seperator=cfg["sources"]["seperator"])
    # unique_protein_names = set([key for key in target_product_map.keys()])

    # with open_conn("../../DB/database.example.ini") as conn:
    #     cur = conn.cursor()
    #     # init names table for faster search
    #     cur.execute("CREATE TEMP TABLE temp_names(name varchar(50));")
    #     cur.executemany("INSERT INTO temp_names (name) VALUES (%s);", [(name,) for name in unique_protein_names])
        
    #     sql = """ 
    #         SELECT protein_id, preferred_name
    #         FROM items.proteins
    #         JOIN temp_names n ON n.name = preferred_name;
    #     """ 
    #     cur.execute(sql)

    #     rows = cur.fetchall()

    #     drug_map = {}
    #     for row in rows:
    #         drug_map[row[0]] = target_product_map[row[1]]

    #     buf = StringIO()
    #     for key, value in drug_map.items():
    #         for drugItem in value:
    #             drugName = drugItem["name"]
    #             drugBankID = drugItem["drugBankID"]
    #             buf.write("{}\t{}\t{}\n".format(key, drugName, drugBankID))

    #     insert_rows_copy_from_factory(conn, "items.drugs")(buf)

    #     conn.commit()
    #     conn.close()


{'main': {'name': 'George Pantziarka TP53 Trust (Created by Anticancer Fund)', 'url': 'https://data.tp53.org.uk/cancerdrugsdb.txt', 'wanted_columns': ['Targets', 'Product', 'DrugBank ID'], 'reversed': True, 'seperator': ';'}, 'extra': [{'name': 'George Pantziarka TP53 Trust (Created by Anticancer Fund)', 'url': 'https://data.tp53.org.uk/cancerdrugsdb.txt', 'wanted_columns': ['Targets', 'Product', 'DrugBank ID'], 'reversed': True, 'seperator': ';'}]}
{'name': 'George Pantziarka TP53 Trust (Created by Anticancer Fund)', 'url': 'https://data.tp53.org.uk/cancerdrugsdb.txt', 'wanted_columns': ['Targets', 'Product', 'DrugBank ID'], 'reversed': True, 'seperator': ';'}: {'name': 'George Pantziarka TP53 Trust (Created by Anticancer Fund)', 'url': 'https://data.tp53.org.uk/cancerdrugsdb.txt', 'wanted_columns': ['Targets', 'Product', 'DrugBank ID'], 'reversed': True, 'seperator': ';'}
https://data.tp53.org.uk/cancerdrugsdb.txt
